# Descargar librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Leer y separar archivo

In [2]:
df = pd.read_csv('../data/logs.csv')
df = df.drop(columns=['_ignored'])
df_llm = df[df['_index'].str.startswith('llm-logs', na=False)]
df_llm = df_llm.drop(columns=['http_status_code', 'service_id', 'client_ip'])
df_system = df[df['_index'].str.startswith('sap-logs', na=False)]
df_system = df_system.drop(columns=['llm_error_message', 'llm_response_size_bytes', 'sap_llm_response_size', 'sap_llm_response_time', 'llm_prompt_tokens', 'llm_completion_tokens', 'llm_total_tokens', 'llm_response_time_ms', 'llm_model_id', 'llm_status', 'llm_cost_usd', 'llm_prompt_category', 'llm_temperature', 'llm_top_p', 'llm_stream', 'llm_finish_reason', 'llm_prompt_id', 'llm_provider', 'llm_prompt'])

# System

In [3]:
df_system.columns

Index(['_id', '_index', '_score', '@timestamp', '@version',
       'event_code_version', 'event_hash', '@event_time_requested',
       'headers_content_type', 'headers_http_host',
       'headers_http_request_method', 'heathers_request_path',
       'sap_function_application', 'sap_source_type', 'sap_function_log_type',
       'sap_function_message', 'sap_app_env', 'region_id', 'region_name',
       'region_code', 'macro_region', 'service_id', 'http_status_code',
       'client_ip'],
      dtype='str')

In [4]:
lista_fila = df_system.iloc[0].tolist()
display(lista_fila)

['23bb795f-593c-4f5b-95f6-9ed2ee49d447',
 'sap-logs-2026.04',
 np.float64(0.7863),
 '2026-04-17T22:12:18.000Z',
 np.int64(1),
 '1.8.7',
 'bb746db61f0d38437ff6e0e4a3a425399f925aa0333352aecc14bb96853843b2',
 '2026-04-17T22:12:17.748Z',
 'application/json',
 'au-syd2.sap-services.internal',
 'GET',
 '/api/sap-fiori-launchpad/status',
 'sap-fiori-launchpad',
 'REST',
 'INFO',
 'Semantic cache hit — previous LLM response reused for sap-fiori-launchpad via REST.',
 'development',
 'AU-003',
 'Australia | Sydney (2)',
 'AU-SYD2',
 'Australia',
 'analytics-engine',
 np.float64(200.0),
 '208.223.2.195']

Posibles columnas para las ventanas de 5 minutos:

- total_requests (Conteo de logs)
- request_rate_std (Desv. Est. del tiempo que hay entre solicitudes consecutivas)
- request_rate_avg (Promedio del tiempo que hay entre solicitudes consecutivas)
- unique_ips_count (Cantidad de ips diferentes. Obtenido de client_ip)

--

- error_rate_400 (Porcentaje sobre total de códigos HTTP que no fueron 200. Obtenido de http_status_code)
- error_rate_500 (Porcentaje sobre total de códigos HTTP que no fueron 200. Obtenido de http_status_code)

--

- ratio_error (Porcentaje sobre el total de log_type. Obtenido de sap_function_log_type)
- ratio_warn (Porcentaje sobre el total de log_type. Obtenido de sap_function_log_type)
- ratio_security (Porcentaje sobre el total de log_type. Obtenido de sap_function_log_type)
- ratio_info (Porcentaje sobre el total de log_type. Obtenido de sap_function_log_type)

--

- ip_entropy (Qué tan balanceadas están las cantidades de solicitudes de las diferentes ips. Obtenido de client_ip)
- unique_paths_ratio (Razón entre el total de rutas diferentes sobre el total de logs. Obtenido de heathers_request_path)
- get_vs_post_ratio (Porcentaje de métodos de tipo "GET" contra el total. Obtenido de headers_http_request_method.)

Además de estás, estaría interesante agregar algo como "invalid_transitions" para contar secuencias de solicitudes de una misma ip que no tengan sentido (como ingresar a algo sin haber pasado por la autenticación) o "sensitive_paths" que cuente cuántas veces se ingresó o trató de ingresar a una ruta importante del sistema (picos a rutas no convencionales podrían significar un ataque).

## Generar ventanas

In [5]:
window_size = pd.Timedelta(minutes=5)
window_step = pd.Timedelta(minutes=2.5)

# Se usa @timestamp como tiempo del evento para construir las ventanas.
required_columns = [
    '@timestamp',
    'http_status_code',
    'client_ip',
    'sap_function_log_type',
    'heathers_request_path',
    'headers_http_request_method',
]
logs = df_system[required_columns].copy()
logs['timestamp'] = pd.to_datetime(logs['@timestamp'], utc=True, errors='coerce')

if logs['timestamp'].isna().any():
    invalid_count = logs['timestamp'].isna().sum()
    raise ValueError(f'Hay {invalid_count} logs sin un @timestamp válido.')

logs['http_status_code'] = pd.to_numeric(logs['http_status_code'], errors='coerce')
logs['log_type'] = logs['sap_function_log_type'].astype('string').str.upper()
logs['method'] = logs['headers_http_request_method'].astype('string').str.upper()

# Cada log pertenece a dos ventanas contiguas con traslape de 2.5 minutos.
window_starts = logs['timestamp'].dt.floor(window_step)
logs['_event_id'] = np.arange(len(logs))
events = pd.concat(
    [
        logs.assign(window_start=window_starts),
        logs.assign(window_start=window_starts - window_size + window_step),
    ],
    ignore_index=True,
)

# Si el último tramo observable dura menos de la mitad de una ventana,
# se integra a la ventana anterior para evitar una muestra desbalanceada.
last_window_start = events['window_start'].max()
if logs['timestamp'].max() - last_window_start < window_step:
    events.loc[events['window_start'].eq(last_window_start), 'window_start'] = (
        last_window_start - window_step
    )
    events = events.drop_duplicates(subset=['window_start', '_event_id'])

status = events['http_status_code']
log_type = events['log_type']
events['is_non_200'] = status.notna() & status.ne(200)
events['is_4xx'] = status.between(400, 499).fillna(False)
events['is_5xx'] = status.between(500, 599).fillna(False)
events['has_log_type'] = log_type.notna()
events['is_error'] = log_type.eq('ERROR').fillna(False)
events['is_warning'] = log_type.eq('WARNING').fillna(False)
events['is_security'] = log_type.eq('SECURITY').fillna(False)
events['is_info'] = log_type.eq('INFO').fillna(False)
events['is_get'] = events['method'].eq('GET').fillna(False)

grouped = events.groupby('window_start', sort=True)
df_batch_system = grouped.agg(
    total_requests=('timestamp', 'size'),
    unique_ips_count=('client_ip', 'nunique'),
    unique_paths_count=('heathers_request_path', 'nunique'),
    non_200_count=('is_non_200', 'sum'),
    count_4xx=('is_4xx', 'sum'),
    count_5xx=('is_5xx', 'sum'),
    log_type_count=('has_log_type', 'sum'),
    count_error=('is_error', 'sum'),
    count_warn=('is_warning', 'sum'),
    count_security=('is_security', 'sum'),
    count_info=('is_info', 'sum'),
    count_get=('is_get', 'sum'),
)

# Los intervalos entre solicitudes consecutivas se calculan dentro de cada ventana, en segundos.
events_sorted = events.sort_values(['window_start', 'timestamp'], kind='mergesort')
intervals = events_sorted.groupby('window_start', sort=False)['timestamp'].diff().dt.total_seconds()
interval_stats = intervals.groupby(events_sorted['window_start']).agg(['mean', 'std'])
df_batch_system['request_rate_avg'] = interval_stats['mean']
df_batch_system['request_rate_std'] = interval_stats['std']

# Tasas como fracciones [0, 1]; el denominador HTTP incluye códigos distintos de 200.
non_200_denominator = df_batch_system['non_200_count'].replace(0, np.nan)
df_batch_system['error_rate_400'] = df_batch_system['count_4xx'] / non_200_denominator
df_batch_system['error_rate_500'] = df_batch_system['count_5xx'] / non_200_denominator
log_type_denominator = df_batch_system['log_type_count'].replace(0, np.nan)
df_batch_system['ratio_error'] = df_batch_system['count_error'] / log_type_denominator
df_batch_system['ratio_warn'] = df_batch_system['count_warn'] / log_type_denominator
df_batch_system['ratio_security'] = df_batch_system['count_security'] / log_type_denominator
df_batch_system['ratio_info'] = df_batch_system['count_info'] / log_type_denominator
df_batch_system['unique_paths_ratio'] = (
    df_batch_system['unique_paths_count'] / df_batch_system['total_requests']
)
df_batch_system['get_vs_post_ratio'] = (
    df_batch_system['count_get'] / df_batch_system['total_requests']
)

# Entropía de Shannon normalizada: 0 = una IP; 1 = solicitudes uniformes entre IPs.
ip_counts = (
    events.dropna(subset=['client_ip'])
    .groupby(['window_start', 'client_ip'], observed=True)
    .size()
)
ip_probabilities = ip_counts / ip_counts.groupby(level=0).transform('sum')
shannon_entropy = -(ip_probabilities * np.log2(ip_probabilities)).groupby(level=0).sum()
unique_ips_per_window = ip_counts.groupby(level=0).size()
normalized_entropy = shannon_entropy.div(np.log2(unique_ips_per_window)).where(
    unique_ips_per_window > 1, 0
)
df_batch_system['ip_entropy'] = normalized_entropy.reindex(df_batch_system.index).fillna(0)

# El índice identifica el inicio de ventana; se eliminan contadores intermedios.
df_batch_system = df_batch_system.drop(
    columns=[
        'unique_paths_count', 'non_200_count', 'count_4xx', 'count_5xx',
        'log_type_count', 'count_error', 'count_warn', 'count_security',
        'count_info', 'count_get',
    ]
)
df_batch_system.index.name = 'window_start'
df_batch_system = df_batch_system[
    [
        'total_requests', 'request_rate_std', 'request_rate_avg',
        'unique_ips_count', 'error_rate_400', 'error_rate_500',
        'ratio_error', 'ratio_warn', 'ratio_security', 'ratio_info',
        'ip_entropy', 'unique_paths_ratio', 'get_vs_post_ratio',
    ]
]

display(df_batch_system.head())

,total_requests,request_rate_std,request_rate_avg,unique_ips_count,error_rate_400,error_rate_500,ratio_error,ratio_warn,ratio_security,ratio_info,ip_entropy,unique_paths_ratio,get_vs_post_ratio
window_start,,,,,,,,,,,,,
2026-04-10 23:57:30+00:00,302,0.705319,0.495017,100,0.276119,0.164179,0.119205,0.178808,0.023179,0.413907,0.971560,0.218543,0.225166
2026-04-11 00:00:00+00:00,594,0.697776,0.504216,105,0.332061,0.152672,0.124579,0.190236,0.023569,0.402357,0.981192,0.112795,0.207071
2026-04-11 00:02:30+00:00,586,0.649259,0.511111,105,0.384314,0.133333,0.119454,0.203072,0.025597,0.383959,0.981451,0.114334,0.186007
2026-04-11 00:05:00+00:00,546,0.637322,0.548624,104,0.375527,0.122363,0.128205,0.186813,0.02381,0.393773,0.982587,0.122711,0.184982
2026-04-11 00:07:30+00:00,566,0.661462,0.529204,104,0.363636,0.150198,0.136042,0.187279,0.024735,0.393993,0.981497,0.118375,0.189046


## Convertir a CSV

In [6]:
df_batch_system['window_index'] = range(1, len(df_batch_system) + 1)
df_batch_system.to_csv('../data/ventanas_separadas/batch_system_indexed.csv', index=True, float_format='%.6f')

# LLM

In [7]:
df_llm.columns

Index(['_id', '_index', '_score', '@timestamp', '@version',
       'event_code_version', 'event_hash', '@event_time_requested',
       'headers_content_type', 'headers_http_host',
       'headers_http_request_method', 'heathers_request_path',
       'sap_function_application', 'sap_source_type', 'sap_function_log_type',
       'sap_function_message', 'sap_app_env', 'sap_llm_response_size',
       'sap_llm_response_time', 'region_id', 'region_name', 'region_code',
       'macro_region', 'llm_model_id', 'llm_provider', 'llm_prompt_id',
       'llm_prompt_category', 'llm_prompt', 'llm_prompt_tokens',
       'llm_completion_tokens', 'llm_total_tokens', 'llm_response_time_ms',
       'llm_response_size_bytes', 'llm_status', 'llm_error_message',
       'llm_cost_usd', 'llm_temperature', 'llm_top_p', 'llm_stream',
       'llm_finish_reason'],
      dtype='str')

In [8]:
lista_fila = df_llm.iloc[0].tolist()
display(lista_fila)

['75225251-b3d5-4a76-97dc-9fcc5f29eeba',
 'llm-logs-2026.04',
 np.float64(0.953),
 '2026-04-12T12:57:39.000Z',
 np.int64(1),
 '1.8.9',
 'ad44a6f8dc6cbad0036218cbc65ae386ab53c846bfe6864b9053cef340af8ad7',
 '2026-04-12T12:57:38.340Z',
 'application/xml',
 'llm.nl-eem.sap-ai.internal',
 'POST',
 '/api/llm/sap-successfactors/completion',
 'sap-successfactors',
 'RFC',
 'LLM_TIMEOUT',
 'LLM TIMEOUT: Request timed out after 30 s. (model=gpt-5.4, app=sap-successfactors)',
 'qa',
 np.float64(nan),
 np.float64(31911.58),
 'EU-023',
 'Netherlands | Eemshaven',
 'NL-EEM',
 'Europe',
 'gpt-5.4',
 'OpenAI',
 'supply-chain-shortage-alert',
 'Supply Chain',
 'A critical component shortage has been flagged for sap-successfactors in Netherlands | Eemshaven. Draft an escalation plan and identify alternative suppliers.',
 np.float64(1451.0),
 np.float64(1229.0),
 np.float64(2680.0),
 np.float64(31911.58),
 np.float64(nan),
 'timeout',
 'Request timed out after 30 s.',
 np.float64(0.0536),
 np.float64(0.2

Posibles columnas para las ventanas de 5 minutos:
- total_llm_requests (Conteo total de logs)
- avg_llm_cost_usd (Promedio de los costos de las llamadas)
- total_llm_cost_usd (Suma de los costos de las llamadas)
- avg_token_ratio (llm_completion_tokens / llm_prompt_tokens. Para comparar el tamaño de los prompts de entrada con los de salida)
- avg_response_time_ms (Promedio de lo que tarda en responder el llm. Obtenido de llm_response_time_ms)

--

- llm_error_rate (Porcentaje de peticiones con llm_status = "error")
- llm_timeout_rate (Porcentaje de peticiones con llm_status = "timeout")

--

- temperature_variance (Qué tan dispersas están las temperaturas de las llamadas. Obtenido de llm_temperature)
- prompt_category_entropy (Entropía de Shannon normalizada entre 0 y 1; 1 indica categorías igualmente frecuentes. Obtenido de llm_prompt_category)
- stream_ratio (Porcentaje de llamadas en las que llm_stream = True)

--

- avg_prompt_length (Longitud promedio de los prompts. Obtenido de llm_prompt)
- max_prompt_length (Longitud del prompt más largo. Obtenido de llm_prompt)


Además de estas variables, estaría interesante agregar algo como "count_math_sus_words" donde tengamos una lista de palabras sospechosas y se contara cuántas veces aparecieron esas palabras en los prompts (como "bypass", "ignore al previous instructions", "give me the password"...)

In [9]:
window_size = pd.Timedelta(minutes=5)
window_step = pd.Timedelta(minutes=2.5)

required_columns = [
    '@timestamp',
    'llm_cost_usd',
    'llm_completion_tokens',
    'llm_prompt_tokens',
    'llm_response_time_ms',
    'llm_status',
    'llm_temperature',
    'llm_prompt_category',
    'llm_stream',
    'llm_prompt',
]
logs = df_llm[required_columns].copy()
logs['timestamp'] = pd.to_datetime(logs['@timestamp'], utc=True, errors='coerce')

if logs['timestamp'].isna().any():
    invalid_count = logs['timestamp'].isna().sum()
    raise ValueError(f'Hay {invalid_count} logs sin un @timestamp válido.')

numeric_columns = [
    'llm_cost_usd',
    'llm_completion_tokens',
    'llm_prompt_tokens',
    'llm_response_time_ms',
    'llm_temperature',
]
for column in numeric_columns:
    logs[column] = pd.to_numeric(logs[column], errors='coerce')

logs['status'] = logs['llm_status'].astype('string').str.upper()
logs['token_ratio'] = logs['llm_completion_tokens'].div(
    logs['llm_prompt_tokens'].replace(0, np.nan)
)
logs['prompt_length'] = logs['llm_prompt'].astype('string').str.len()
logs['is_error'] = logs['status'].eq('ERROR').fillna(False)
logs['is_timeout'] = logs['status'].eq('TIMEOUT').fillna(False)
logs['is_stream'] = (
    logs['llm_stream'].astype('string').str.lower().eq('true').fillna(False)
)

# Cada solicitud se asigna a dos ventanas de 5 minutos separadas por 2.5 minutos.
window_starts = logs['timestamp'].dt.floor(window_step)
logs['_event_id'] = np.arange(len(logs))
events = pd.concat(
    [
        logs.assign(window_start=window_starts),
        logs.assign(window_start=window_starts - window_size + window_step),
    ],
    ignore_index=True,
)

# Si el último tramo observable dura menos de la mitad de una ventana,
# se integra a la ventana anterior para evitar una muestra desbalanceada.
last_window_start = events['window_start'].max()
if logs['timestamp'].max() - last_window_start < window_step:
    events.loc[events['window_start'].eq(last_window_start), 'window_start'] = (
        last_window_start - window_step
    )
    events = events.drop_duplicates(subset=['window_start', '_event_id'])

grouped = events.groupby('window_start', sort=True)
df_batch_llm = grouped.agg(
    total_llm_requests=('timestamp', 'size'),
    avg_llm_cost_usd=('llm_cost_usd', 'mean'),
    total_llm_cost_usd=('llm_cost_usd', 'sum'),
    avg_token_ratio=('token_ratio', 'mean'),
    avg_response_time_ms=('llm_response_time_ms', 'mean'),
    llm_error_rate=('is_error', 'mean'),
    llm_timeout_rate=('is_timeout', 'mean'),
    temperature_variance=('llm_temperature', 'var'),
    stream_ratio=('is_stream', 'mean'),
    avg_prompt_length=('prompt_length', 'mean'),
    max_prompt_length=('prompt_length', 'max'),
)

# Normalized Shannon entropy is 0 for one category and 1 for a uniform distribution.
category_counts = (
    events.dropna(subset=['llm_prompt_category'])
    .groupby(['window_start', 'llm_prompt_category'], observed=True)
    .size()
)
category_probabilities = (
    category_counts / category_counts.groupby(level=0).transform('sum')
)
category_entropy = -(
    category_probabilities * np.log2(category_probabilities)
).groupby(level=0).sum()
categories_per_window = category_counts.groupby(level=0).size()
normalized_category_entropy = category_entropy.div(
    np.log2(categories_per_window)
).where(categories_per_window > 1, 0)
df_batch_llm['prompt_category_entropy'] = (
    normalized_category_entropy.reindex(df_batch_llm.index).fillna(0)
)

df_batch_llm.index.name = 'window_start'
df_batch_llm = df_batch_llm[
    [
        'total_llm_requests',
        'avg_llm_cost_usd',
        'total_llm_cost_usd',
        'avg_token_ratio',
        'avg_response_time_ms',
        'llm_error_rate',
        'llm_timeout_rate',
        'temperature_variance',
        'prompt_category_entropy',
        'stream_ratio',
        'avg_prompt_length',
        'max_prompt_length',
    ]
]

display(df_batch_llm.head())

,total_llm_requests,avg_llm_cost_usd,total_llm_cost_usd,avg_token_ratio,avg_response_time_ms,llm_error_rate,llm_timeout_rate,temperature_variance,prompt_category_entropy,stream_ratio,avg_prompt_length,max_prompt_length
window_start,,,,,,,,,,,,
2026-04-10 23:57:30+00:00,193,0.011026,2.128112,1.396312,8287.870881,0.181347,0.072539,0.180711,0.951361,0.502591,112.025907,174
2026-04-11 00:00:00+00:00,394,0.012840,5.058822,1.313032,8720.057462,0.195431,0.093909,0.177682,0.959043,0.51269,110.560914,174
2026-04-11 00:02:30+00:00,417,0.013249,5.524788,1.262083,8502.081415,0.211031,0.083933,0.177055,0.974366,0.520384,111.179856,173
2026-04-11 00:05:00+00:00,396,0.011730,4.645221,1.335657,7923.143106,0.219697,0.070707,0.186211,0.985586,0.494949,111.194444,169
2026-04-11 00:07:30+00:00,391,0.012933,5.056919,1.424676,8657.928491,0.235294,0.117647,0.184881,0.976698,0.514066,110.296675,169


## Convertir a CSV

In [10]:
df_batch_llm['window_index'] = range(1, len(df_batch_llm) + 1)
df_batch_llm.to_csv('../data/ventanas_separadas/batch_llm_indexed.csv', index=True, float_format='%.6f')